In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score
from sklearn.model_selection import RandomizedSearchCV
from skorch import NeuralNetClassifier
import joblib

In [4]:
X_train = pd.read_csv("csv_files/X_train.csv")
X_test = pd.read_csv("csv_files/X_test.csv")
y_train = pd.read_csv("csv_files/y_train_encoded.csv").values.astype(np.int64).ravel()
y_test = pd.read_csv("csv_files/y_test_encoded.csv").values.astype(np.int64).ravel()

In [5]:
class NNClassifier(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(NNClassifier, self).__init__()
        self.fc1 = nn.Linear(input_dim, 64)  # input_dim is the number of input features
        self.fc2 = nn.Linear(64, 32)  # 64 neurons to 32 neurons
        self.fc3 = nn.Linear(32, output_dim)   # 32 neurons to 4 output_dim because we have 4 classes


    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.fc3(x)

### Why Neurons are Halved in Neural Network Layers

1. **Dimensionality Reduction**  
   - Encourages learning abstract, compressed features.
   - Creates a bottleneck to focus on the most relevant information.

2. **Computational Efficiency**  
   - Fewer neurons = fewer parameters, reducing overfitting risk.
   - Balances network complexity and speeds up computation.

3. **Empirical Success**  
   - Common in successful architectures; works well for many tasks.

4. **Compact Representations**  
   - Forces the network to learn meaningful, compact data representations.

5. **Symmetry and Simplicity**  
   - Powers of 2 (e.g., 64, 32, 16) align well with binary data and create simple, effective designs.


In [6]:
# Instantiate the model
net = NeuralNetClassifier(
    NNClassifier,
    criterion=nn.CrossEntropyLoss, # Standard loss function for multi-class classification task
    optimizer=optim.Adam, # Adapts the learning rate for each parameter, leading to faster convergence and better performance
    max_epochs=100,  # Static number of epochs
    batch_size=2,  # Static batch size
)

In [7]:
# Define the pipeline
scaler = MinMaxScaler()
pipe = Pipeline([
    ('scaler', scaler),
    ('net', net),
])

In [8]:
# Hyperparameters for grid search
param_grid = {
    'lr': [0.01, 0.001, 0.0001]
}


# Perform randomized grid search
random_search = RandomizedSearchCV(net, param_grid, n_iter=10, cv=10, verbose=2)
random_search.fit(X_train, y_train)

d:\Projects\GEOAI\GeoAI-ISPRS-SS\geoai-env\Lib\site-packages\sklearn\model_selection\_search.py:320: UserWarning: The total space of parameters 3 is smaller than n_iter=10. Running 3 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Fitting 10 folds for each of 3 candidates, totalling 30 fits
[CV] END ............................................lr=0.01; total time=   0.0s
[CV] END ............................................lr=0.01; total time=   0.0s
[CV] END ............................................lr=0.01; total time=   0.0s
[CV] END ............................................lr=0.01; total time=   0.0s
[CV] END ............................................lr=0.01; total time=   0.0s
[CV] END ............................................lr=0.01; total time=   0.0s
[CV] END ............................................lr=0.01; total time=   0.0s
[CV] END ............................................lr=0.01; total time=   0.0s
[CV] END ............................................lr=0.01; total time=   0.0s
[CV] END ............................................lr=0.01; total time=   0.0s
[CV] END ...........................................lr=0.001; total time=   0.0s
[CV] END .......................................

ValueError: 
All the 30 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
30 fits failed with the following error:
Traceback (most recent call last):
  File "d:\Projects\GEOAI\GeoAI-ISPRS-SS\geoai-env\Lib\site-packages\sklearn\model_selection\_validation.py", line 888, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "d:\Projects\GEOAI\GeoAI-ISPRS-SS\geoai-env\Lib\site-packages\skorch\classifier.py", line 165, in fit
    return super(NeuralNetClassifier, self).fit(X, y, **fit_params)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\Projects\GEOAI\GeoAI-ISPRS-SS\geoai-env\Lib\site-packages\skorch\net.py", line 1317, in fit
    self.initialize()
  File "d:\Projects\GEOAI\GeoAI-ISPRS-SS\geoai-env\Lib\site-packages\skorch\net.py", line

In [7]:
# Get the best model
best_model = random_search.best_estimator_

print(f"Best model during training: {random_search.best_score_}")

# Save the model's state_dict
torch.save(best_model.module_.state_dict(), 'best_model_state_dict.pth')
joblib.dump(random_search, 'random_search.pkl')


Best model during training: 0.9361595077913863


['random_search.pkl']

In [8]:
# Evaluate on the test set
y_pred = best_model.predict(X_test_scaled)
accuracy = accuracy_score(y_test, y_pred)
print(f'Best model accuracy: {accuracy:.4f}')

Best model accuracy: 0.9399
